# Vehicle Detector Training — Traffic Analytics System

Fine-tune **YOLO11m** on UA-DETRAC traffic data (~20K train, 4 classes: **bus, car, truck, van**).

**Why this matters:** Better vehicle boxes → cleaner crops for **plate OCR** and **tracking** on Indian/highway CCTV.

**Motorcycles:** This dataset has no motorcycle class. After training, the local pipeline uses **COCO yolo11s fallback** for bikes (helmet detection).

## Kaggle session rules

1. Add input dataset **`traffic-vehicle-balanced`** in the sidebar.
2. Settings → **GPU T4 x2** (or T4 x1), **Internet ON**.
3. **Save Version → Save & Run All** before leaving — training takes **~4–8 hours** (100 epochs).
4. Download `vehicle_detector.pt` from **Output** after epoch 10+ as backup.

In [ ]:
# --- CONFIG ---
DATASET_SLUG = "traffic-vehicle-balanced"
WEIGHTS_SLUG = "traffic-vehicle-weights"

CONTINUE_FROM_INPUT = False
RESUME = False

BASE_WEIGHTS = "yolo11m.pt"
EPOCHS = 100
BATCH = 8                     # use 4 if CUDA OOM on T4
IMGSZ = 640
PATIENCE = 15                 # early stop if val mAP plateaus
CLOSE_MOSAIC = 10
WORKERS = 2
SEED = 42
SAVE_PERIOD = 1
USE_WRITABLE_CACHE = True

RUN_NAME = "vehicle_detector"
PROJECT = "traffic_analytics"

In [ ]:
import shutil
import zipfile
from pathlib import Path

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable GPU: Settings -> Accelerator -> GPU T4")

!pip install -q ultralytics>=8.3.0

In [ ]:
import os
from pathlib import Path


def find_vehicle_root() -> Path:
    input_dir = Path("/kaggle/input")
    candidates = [
        input_dir / DATASET_SLUG / "vehicle_detection",
        input_dir / DATASET_SLUG,
        input_dir / "vehicledetection" / "vehicle_detection",
    ]
    for path in candidates:
        if (path / "train" / "images").is_dir():
            return path
    for path in input_dir.rglob("vehicle_detection"):
        if (path / "train" / "images").is_dir():
            return path
    raise FileNotFoundError(f"vehicle_detection not found under {input_dir}")


def prepare_writable_dataset(input_root: Path, working: Path) -> Path:
    cache_root = working / "vehicle_cached"
    train_cache = cache_root / "train" / "labels" / "train.cache"

    if train_cache.is_file():
        print("Writable label cache found:", train_cache)
        return cache_root

    print("Building writable dataset (~3–8 min)...")
    for split in ("train", "valid", "test"):
        src = input_root / split
        dst_img = cache_root / split / "images"
        dst_lbl = cache_root / split / "labels"
        dst_lbl.mkdir(parents=True, exist_ok=True)

        if not dst_img.exists():
            os.symlink(src / "images", dst_img, target_is_directory=True)

        if src.joinpath("labels").is_dir() and not any(dst_lbl.glob("*.txt")):
            print(f"  copying labels: {split}")
            for lbl in (src / "labels").glob("*.txt"):
                shutil.copy2(lbl, dst_lbl / lbl.name)

    return cache_root


INPUT_ROOT = find_vehicle_root()
print("Found dataset at:", INPUT_ROOT)

WORKING = Path("/kaggle/working")
RUN_DIR = WORKING / "runs" / "detect" / PROJECT / RUN_NAME
LAST_PT = RUN_DIR / "weights" / "last.pt"
BEST_PT = RUN_DIR / "weights" / "best.pt"
OUTPUT_MODEL = WORKING / "vehicle_detector.pt"
BACKUP_MODEL = WORKING / "vehicle_detector_backup.pt"

DATA_ROOT = prepare_writable_dataset(INPUT_ROOT, WORKING) if USE_WRITABLE_CACHE else INPUT_ROOT

DATA_YAML = WORKING / "data.yaml"
DATA_YAML.write_text(f"""path: {DATA_ROOT}
train: train/images
val: valid/images
test: test/images
nc: 4
names:
  - bus
  - car
  - truck
  - van
""")

for split in ("train", "valid", "test"):
    img_dir = DATA_ROOT / split / "images"
    n = sum(1 for _ in img_dir.iterdir()) if img_dir.is_dir() else 0
    print(f"{split}: {n} images")

print("\nCheckpoints:")
for label, path in [("last", LAST_PT), ("best", BEST_PT), ("backup", BACKUP_MODEL)]:
    print(f"  {label}: {'YES' if path.is_file() else 'no'} {path}")

In [ ]:
from ultralytics import YOLO

if RESUME and LAST_PT.is_file():
    weights, mode = str(LAST_PT), "resume"
elif RESUME and BACKUP_MODEL.is_file():
    weights, mode = str(BACKUP_MODEL), "resume-from-backup"
elif CONTINUE_FROM_INPUT:
    candidate = Path("/kaggle/input") / WEIGHTS_SLUG / "vehicle_detector.pt"
    if candidate.is_file():
        weights, mode = str(candidate), "continue-best"
    else:
        weights, mode = BASE_WEIGHTS, "fresh"
else:
    weights, mode = BASE_WEIGHTS, "fresh"

print(f"Mode: {mode}")
print(f"Weights: {weights}")


def backup_best(trainer):
    src = Path(trainer.best) if trainer.best else BEST_PT
    if src.is_file():
        shutil.copy2(src, BACKUP_MODEL)
        shutil.copy2(src, OUTPUT_MODEL)
        print(f"  [backup] epoch {trainer.epoch + 1}: -> {OUTPUT_MODEL.name}")


model = YOLO(weights)
model.add_callback("on_fit_epoch_end", backup_best)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    close_mosaic=CLOSE_MOSAIC,
    project=str(WORKING / "runs" / "detect" / PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    device=0,
    amp=True,
    workers=WORKERS,
    save_period=SAVE_PERIOD,
    seed=SEED,
    cos_lr=True,
    mosaic=1.0,
    plots=True,
    val=True,
    resume=(mode == "resume"),
    # Indian traffic / CCTV augmentation (Paper 4 + highway scenarios)
    hsv_h=0.015,
    hsv_s=0.70,
    hsv_v=0.50,
    degrees=5.0,
    translate=0.12,
    scale=0.45,
    fliplr=0.5,
)
print("Training finished.")

In [ ]:
if not BEST_PT.is_file():
    BEST_PT = Path(results.save_dir) / "weights" / "best.pt"

assert BEST_PT.is_file(), f"best.pt not found at {BEST_PT}"
shutil.copy2(BEST_PT, OUTPUT_MODEL)
print(f"Saved -> {OUTPUT_MODEL} ({OUTPUT_MODEL.stat().st_size / 1e6:.1f} MB)")

metrics = model.val(data=str(DATA_YAML), split="test")
print("Test mAP50:", metrics.box.map50)
print("Test mAP50-95:", metrics.box.map50_95)

In [ ]:
zip_path = WORKING / "vehicle_training_output.zip"
run_folder = BEST_PT.parent.parent

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in run_folder.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(WORKING))
    zf.write(OUTPUT_MODEL, OUTPUT_MODEL.name)

print("Download from Output:")
print(f"  - {OUTPUT_MODEL.name}")
print(f"  - {zip_path.name}")